# 05 - Visualizzazione Territoriale della Capitanata

In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'test-download-capitanata'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

# caricamento del dataset e della legenda
dataset_path = PROCESSED_DIR / "dataset" / "dataset.parquet"
# points_path = INTERIM_DIR / "points.json"
legend_path = PROCESSED_DIR / "legend.json"

df_dataset = pd.read_parquet(dataset_path)

with open(legend_path, "r", encoding="utf-8") as f:
    legend = json.load(f)

print(f"Dataset caricato con {len(df_dataset)} campi.")
print(f"Legenda disponibile con {len(legend)} classi colturali.")

#### Predizione dell'intera Capitanata
flattening dell'immagine -> predizione -> reshape mappa

In [ ]:
import joblib
from src.feature_engineering import (                        
    FEATURE_LIST,                                           
    load_monthly_capitanata_rasters,                         
    extract_features_from_monthly_stack                      
)                              

# caricamento del modello
MODELS_DIR = DATA_DIR / "models"
model_path = MODELS_DIR / "random_forest_crop_model.joblib"
model = joblib.load(model_path)
print("Modello caricato!")

# caricamento dei 12 TIF mensili della Capitanata                          
capitanata_dir = PROCESSED_DIR / "sentinel2_capitanata_area" 
monthly_stack, profile = load_monthly_capitanata_rasters(capitanata_dir, year=2023, max_size=1000)                                                 
                                                                
n_months, n_bands, h, w = monthly_stack.shape                
print(f"Stack mensile caricato: {n_months} mesi, {n_bands} bande, risoluzione: {h}x{w} pixel.")                           
                                                                
# estrazione vettoriale delle feature                                                   
_, features_matrix = extract_features_from_monthly_stack(monthly_stack)             
print(f"Matrice feature generata: {features_matrix.shape[0]:,} pixel x {features_matrix.shape[1]} feature.")                        
                                                                
# predizione spaziale su tutti i pixel                    
pred_flat = model.predict(features_matrix)                         
pred_map = pred_flat.reshape(h, w)                           
print("Predizione completata!")  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
import rasterio
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.warp import reproject
from rasterio.crs import CRS
from rasterio.transform import Affine

# Caricamento di un mese della Capitanata e conversione in immagine RGB
image = monthly_stack[5] # Giugno
# Rasterio ordina 1-based: [1:Blu, 2:Verde, 3:Rosso, ...]
# NumPy ordina 0-based: [0:Blu, 1:Verde, 2:Rosso, ...]
rgb_img = np.dstack((image[2], image[1], image[0]))

# normalizzazione per visualizzazione
rgb_img = rgb_img / np.percentile(rgb_img, 98)
rgb_img = np.clip(rgb_img, 0, 1)

# -------------------------------------------------------------------
# Recupero dei colori ufficiali Copernicus
clr_files = list(RAW_DIR.rglob("*.clr"))
color_map = {}

if clr_files:
    with open(clr_files[0], "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                code = int(parts[0])
                if code != 65535: # escludiamo il nodata
                    color_map[code] = [
                        int(parts[1]) / 255.0,
                        int(parts[2]) / 255.0,
                        int(parts[3]) / 255.0
                    ]

# -------------------------------------------------------------------
# Fusione e riproiezione della ground truth (Copernicus CLMS) sulla stessa griglia UTM di Sentinel-2
copernicus_dir = PROCESSED_DIR / "crops_types_yearly_capitanata_4326" / "2023"
tif_files = sorted(list(copernicus_dir.rglob("*.tif")))

src_files = [rasterio.open(f) for f in tif_files]
mosaic, out_trans = merge(src_files, res=(0.001, 0.001), resampling=Resampling.nearest)
mosaic = mosaic.squeeze()
for s in src_files:
    s.close()

# adattamento della trasformazione geometrica alle dimensioni attuali (h, w)
h, w = pred_map.shape
sx = profile["width"] / w
sy = profile["height"] / h
dst_transform = profile["transform"] * Affine.scale(sx, sy)

gt_map = np.zeros((h, w), dtype=np.uint16)
reproject(
    source=mosaic,
    destination=gt_map,
    src_transform=out_trans,
    src_crs=CRS.from_epsg(4326),
    dst_transform=dst_transform,
    dst_crs=profile["crs"],
    resampling=Resampling.nearest,
    src_nodata=65535,
    dst_nodata=0
)

# colorazione della mappa di ground truth
gt_rgb = np.ones((h, w, 3), dtype=np.float32) * 0.95 # sfondo grigio chiaro per aree non agricole (boschi, città)

for code, color in color_map.items():
    if code != 0:
        mask = (gt_map == code)
        if np.any(mask):
            gt_rgb[mask] = color

# azzurro del mare
mask_water = (gt_map == 65534)
gt_rgb[mask_water] = [0.85, 0.92, 0.98]

# -------------------------------------------------------------------
# Colorazione della mappa predetta con maschera agricola (Cropland Mask)
pred_rgb = np.ones((h, w, 3), dtype=np.float32) * 0.95 # sfondo grigio chiaro per aree non agricole

# identificazione veri terreni agricoli da Copernicus (esclusione boschi, città e mare)
valid_crops = [int(k) for k in legend.keys() if int(k) not in [0, 65534, 65535]]
is_cropland = np.isin(gt_map, valid_crops)

# colorazione predizione del modello SOLO dove sono presenti campi agricoli reali
for code, color in color_map.items():
    if code != 0:
        mask = (pred_map == code) & is_cropland
        if np.any(mask):
            pred_rgb[mask] = color

# azzurro del mare
pred_rgb[mask_water] = [0.85, 0.92, 0.98]

# -------------------------------------------------------------------
# Mappa di differenza ed errori di predizione
pred_hsv = rgb_to_hsv(pred_rgb)
pred_hsv[..., 1] *= 0.15 # saturazione della predizione di sfondo ridotta al 15%
diff_rgb = hsv_to_rgb(pred_hsv)
diff_rgb[mask_water] = [0.85, 0.92, 0.98] # azzurro del mare

# pixel dove la predizione diverge dal ground truth (solo su terreni agricoli)
wrong_predicted = (gt_map != pred_map) & is_cropland
diff_rgb[wrong_predicted] = [1, 0, 0] # rosso per gli errori

# calcolo dell'accuratezza spaziale sui campi
correct_pixels = np.sum((gt_map == pred_map) & is_cropland)
total_cropland_pixels = np.sum(is_cropland)
spatial_acc = (correct_pixels / total_cropland_pixels) * 100 if total_cropland_pixels > 0 else 0.0

# -------------------------------------------------------------------
# Creazione del plot a 4 riquadri affiancati
fig, axes = plt.subplots(1, 4, figsize=(28, 7))

# immagine satellitare RGB
axes[0].imshow(rgb_img)
axes[0].set_title('Immagine reale Capitanata (RGB)')
axes[0].axis('off')

# ground truth copernicus
axes[1].imshow(gt_rgb)
axes[1].set_title('Ground Truth Copernicus CLMS (2023)')
axes[1].axis('off')

# predizione rgb mascherata
axes[2].imshow(pred_rgb)
axes[2].set_title('Predizione della Capitanata (Random Forest)')
axes[2].axis('off')

# mappa degli errori
axes[3].imshow(diff_rgb)
axes[3].set_title(f'Errori di predizione (Acc: {spatial_acc:.1f}%)')
axes[3].axis('off')

# legenda dinamica (mostra le classi colturali + indicatore di errore)
unique_classes = sorted(list(set(np.unique(pred_map[is_cropland])).union(set(np.unique(gt_map[is_cropland])))))
legend_handles = [
    Patch(
        facecolor=color_map.get(int(c), [0.5, 0.5, 0.5]),
        edgecolor="black",
        label=legend.get(str(int(c)), f"Classe {c}")
    )
    for c in unique_classes if int(c) in color_map and int(c) not in [0, 65534, 65535]
]
legend_handles.append(
    Patch(facecolor=[0.91, 0.20, 0.20], edgecolor="black", label="Errore di predizione")
)

axes[3].legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    title="Classi colturali",
    title_fontsize=11,
    fontsize=9,
    frameon=True,
    facecolor="white",
    framealpha=0.95
)

plt.tight_layout()
plt.show()
